# 20 Newsgroups Information Retrieval — TF-IDF Retrieval System Demo

**Author:** Reetikesh Choudhury  
**Module:** TF-IDF Retrieval (`src/tfidf.py`)

This notebook demonstrates the **TF-IDF vector retrieval engine** built on the common processed dataset (`data/processed/processed_documents.parquet`).

---

### Pipeline Overview
1. **Load Processed Corpus** (`37,588` documents preprocessed via `src/preprocessing.py`)
2. **Build TF-IDF Index** (`sklearn.feature_extraction.text.TfidfVectorizer` with sublinear scaling)
3. **Execute Natural Language Queries** & compute cosine similarity rankings
4. **Inspect Results** formatted with `doc_id`, `score`, `rank`, and document category metadata.

In [ ]:
import sys
import time
from pathlib import Path
import pandas as pd

# Add src/ directory to Python path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tfidf import build_tfidf, search_tfidf, get_document, load_processed_dataset

## Step 1: Load Common Processed Dataset

Load the preprocessed Parquet artifact generated by `src/build_dataset.py`.

In [ ]:
t0 = time.perf_counter()
df = load_processed_dataset()
load_time = time.perf_counter() - t0

print(f"Dataset loaded in {load_time:.2f} seconds.")
print(f"Total documents: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

df.head()

## Step 2: Build TF-IDF Retrieval Index

Fit the `TfidfVectorizer` on `clean_text` tokens and create the sparse document matrix.

In [ ]:
t0 = time.perf_counter()
searcher = build_tfidf(df)
build_time = time.perf_counter() - t0

print(f"TF-IDF Index built in {build_time:.2f} seconds.")
print(f"Total Documents Indexed: {searcher.num_documents:,}")
print(f"Vocabulary Size:        {searcher.vocab_size:,} terms")

## Step 3: Run Sample Queries Across Newsgroup Topics

Execute queries across various categories (Space, Cryptography, Graphics, Motorcycles, Religion, Windows, Sports) and display top-5 retrieved results.

In [ ]:
sample_queries = [
    "space shuttle launch moon orbit",
    "3d graphics rendering algorithm ray tracing",
    "public key encryption pgp cryptography security",
    "motorcycle helmet riding safety gears",
    "christian church faith jesus religion",
    "windows driver installation system setup",
    "baseball pitcher home run inning game",
]

for query in sample_queries:
    t_start = time.perf_counter()
    results = searcher.search(query, top_k=5)
    search_ms = (time.perf_counter() - t_start) * 1000
    
    print(f"\nQuery: '{query}' ({search_ms:.2f} ms)")
    
    query_display = []
    for res in results:
        doc = get_document(res["doc_id"])
        query_display.append({
            "Rank": res["rank"],
            "doc_id": res["doc_id"],
            "Score": round(res["score"], 4),
            "Category": doc["category"] if doc else "N/A",
            "Snippet": doc["text"].replace("\n", " ").strip()[:50] + "..." if doc else ""
        })
    
    display_df = pd.DataFrame(query_display)
    display(display_df)

## Step 4: Verify Standardized Output Structure

Verify that `search_tfidf()` returns the exact standardized dictionary format expected by the team:

```json
[
    {
        "doc_id": 27983,
        "score": 0.361942,
        "rank": 1
    }
]
```

In [ ]:
# Standalone function call
results = search_tfidf("space exploration", top_k=3)

import json
print(json.dumps(results, indent=2))